In [0]:
start_row = 1     
end_row = 50        
batch_size = 5       
adls_base_path = "abfss://indiaprojectraw@indiaprojectstorage.dfs.core.windows.net/customer"  


from pyspark.sql import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id, ceil, col, lit

df = spark.table("`sales-northindia-db`.saleslt.customer")

window_spec = Window.orderBy(monotonically_increasing_id())
df_with_row = df.withColumn("_row_num", row_number().over(window_spec))

df_filtered = df_with_row.filter(
    (col("_row_num") >= start_row) & (col("_row_num") <= end_row)
)

print(f"Total rows in table : {df.count()}")
print(f"Rows selected ({start_row} to {end_row}): {df_filtered.count()}")

df_batched = df_filtered.withColumn(
    "_batch_id",
    ceil((col("_row_num") - lit(start_row) + lit(1)) / lit(batch_size)).cast("int")
)

total_batches = df_batched.select("_batch_id").distinct().count()
print(f"Number of batches (batch_size={batch_size}): {total_batches}")

# Write all batches once to a temp folder (one partition folder per batch, one file each)
tmp_path = f"{adls_base_path}/_tmp_batches"
df_batched.drop("_row_num") \
    .repartition("_batch_id") \
    .write \
    .mode("overwrite") \
    .partitionBy("_batch_id") \
    .json(tmp_path)

# Move each batch's part file to customerbatch<N>.json, then remove the temp folder
written_files = []
for folder in dbutils.fs.ls(tmp_path):
    if not folder.name.startswith("_batch_id="):
        continue
    batch_id = int(folder.name.rstrip("/").split("=")[1])
    part_files = [f.path for f in dbutils.fs.ls(folder.path) if f.name.startswith("part-") and f.name.endswith(".json")]
    if len(part_files) != 1:
        raise RuntimeError(f"Expected 1 part file for batch {batch_id}, found {len(part_files)}")
    target_file = f"{adls_base_path}/customerbatch{batch_id}.json"
    dbutils.fs.rm(target_file)  # replace file from a previous run, if any
    dbutils.fs.mv(part_files[0], target_file)
    written_files.append(target_file)

dbutils.fs.rm(tmp_path, True)

print(f"✅ {len(written_files)} JSON batch file(s) written to: {adls_base_path}")
for f in sorted(written_files, key=lambda p: int(p.split("customerbatch")[1].split(".")[0])):
    print("  ", f)

# Quick preview of the first batch
display(df_batched.filter(col("_batch_id") == 1).drop("_row_num", "_batch_id"))

In [0]:
start_row = 1     
end_row = 50        
batch_size = 5       
adls_base_path = "abfss://indiaprojectraw@indiaprojectstorage.dfs.core.windows.net/customeraddress"  


from pyspark.sql import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id, ceil, col, lit

df = spark.table("`sales-northindia-db`.saleslt.customeraddress")

window_spec = Window.orderBy(monotonically_increasing_id())
df_with_row = df.withColumn("_row_num", row_number().over(window_spec))

df_filtered = df_with_row.filter(
    (col("_row_num") >= start_row) & (col("_row_num") <= end_row)
)

print(f"Total rows in table : {df.count()}")
print(f"Rows selected ({start_row} to {end_row}): {df_filtered.count()}")

df_batched = df_filtered.withColumn(
    "_batch_id",
    ceil((col("_row_num") - lit(start_row) + lit(1)) / lit(batch_size)).cast("int")
)

total_batches = df_batched.select("_batch_id").distinct().count()
print(f"Number of batches (batch_size={batch_size}): {total_batches}")

# Write all batches once to a temp folder (one partition folder per batch, one file each)
tmp_path = f"{adls_base_path}/_tmp_batches"
df_batched.drop("_row_num") \
    .repartition("_batch_id") \
    .write \
    .mode("overwrite") \
    .partitionBy("_batch_id") \
    .csv(tmp_path)

# Move each batch's part file to customerbatch<N>.json, then remove the temp folder
written_files = []
for folder in dbutils.fs.ls(tmp_path):
    if not folder.name.startswith("_batch_id="):
        continue
    batch_id = int(folder.name.rstrip("/").split("=")[1])
    part_files = [f.path for f in dbutils.fs.ls(folder.path) if f.name.startswith("part-") and f.name.endswith(".csv")]
    if len(part_files) != 1:
        raise RuntimeError(f"Expected 1 part file for batch {batch_id}, found {len(part_files)}")
    target_file = f"{adls_base_path}/customeraddressbatch{batch_id}.csv"
    dbutils.fs.rm(target_file)  # replace file from a previous run, if any
    dbutils.fs.mv(part_files[0], target_file)
    written_files.append(target_file)

dbutils.fs.rm(tmp_path, True)

print(f"✅ {len(written_files)} JSON batch file(s) written to: {adls_base_path}")
for f in sorted(written_files, key=lambda p: int(p.split("customeraddressbatch")[1].split(".")[0])):
    print("  ", f)

# Quick preview of the first batch
display(df_batched.filter(col("_batch_id") == 1).drop("_row_num", "_batch_id"))